In [1]:
%pip install scipy



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import sqlite3

db_path = r"../data/nepse_data.db"


# Open connection safely using a context manager
with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()
    
    # Execute a query
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    
    # Fetch all table names
    tables = cursor.fetchall()
    print("Tables in database:", tables)

    schema = pd.read_sql_query("PRAGMA table_info(scrape_log);", conn)

Tables in database: [('daily_prices',), ('companies',), ('scrape_log',)]


In [27]:
daily_prices = pd.read_sql_query("PRAGMA table_info(daily_prices);", conn)
companies = pd.read_sql_query("PRAGMA table_info(companies);", conn)
scrape_log = pd.read_sql_query("PRAGMA table_info(scrape_log)", conn)

print("daily_price SCHEMA:", daily_prices)
print("companies SCHEMA" ,companies)
print("scrape_log SCHEMA", scrape_log)

daily_price SCHEMA:    cid            name  type  notnull dflt_value  pk
0    0          symbol  TEXT        1       None   1
1    1            date  TEXT        1       None   2
2    2            open  REAL        0       None   0
3    3            high  REAL        0       None   0
4    4             low  REAL        0       None   0
5    5           close  REAL        1       None   0
6    6          volume  REAL        0       None   0
7    7  percent_change  REAL        0       None   0
8    8        turnover  REAL        0       None   0
companies SCHEMA    cid            name  type  notnull dflt_value  pk
0    0          symbol  TEXT        0       None   1
1    1    company_name  TEXT        0       None   0
2    2  promoter_share  TEXT        0       None   0
3    3    public_share  TEXT        0       None   0
4    4    listing_date  TEXT        0       None   0
scrape_log SCHEMA    cid             name  type  notnull dflt_value  pk
0    0           symbol  TEXT        0     

In [53]:
def get_last_data_date_universe(conn: sqlite3.Connection) -> pd.Timestamp:
    last_data_dates_df = pd.read_sql_query("SELECT last_data_date FROM scrape_log;", conn, parse_dates = ["last_data_date"])
    return  last_data_dates_df['last_data_date'].max()


last_data_date = get_last_data_date_universe(conn)
print(last_data_date)

2026-10-16 00:00:00


In [72]:
from typing import Dict
def load_market_data(conn: sqlite3.Connection) -> Dict[str, pd.DataFrame]:
    registy = {}
    symbol_df = pd.read_sql_query ("SELECT DISTINCT symbol FROM daily_prices;", conn)
    
    for sym in symbol_df["symbol"]:
        df = pd.read_sql_query("SELECT * FROM daily_prices WHERE symbol = ? ORDER BY date ASC", conn, params = (sym,), parse_dates = "date")
        registy[sym] = df
    return registy

print(load_market_data(conn))

{'ACLBSL':       symbol       date         open         high          low        close  \
0     ACLBSL 2021-10-18  1210.826446  1219.008264  1177.685950  1190.082645   
1     ACLBSL 2021-10-19  1169.421488  1171.900826  1123.966942  1136.363636   
2     ACLBSL 2021-10-20  1157.024793  1173.553719  1135.537190  1156.198347   
3     ACLBSL 2021-10-21  1141.322314  1161.157025  1141.322314  1153.801653   
4     ACLBSL 2021-10-24  1157.024793  1180.165289  1152.892562  1180.165289   
...      ...        ...          ...          ...          ...          ...   
1154  ACLBSL 2026-10-12   857.000000   860.000000   853.000000   853.000000   
1155  ACLBSL 2026-10-13   852.000000   852.300000   852.000000   852.000000   
1156  ACLBSL 2026-10-14   850.000000   865.000000   845.500000   846.000000   
1157  ACLBSL 2026-10-15   849.000000   849.000000   840.000000   840.000000   
1158  ACLBSL 2026-10-16   850.000000   855.000000   850.000000   850.000000   

      volume  percent_change turnover  
